# NB_ingest_to_hubs — Silver → Hubs

Reads each Silver source table defined in the DV model config,
computes the SHA-256 hash key from business key columns, and performs
an INSERT-only MERGE into the corresponding vault hub table.
No updates — hubs are insert-only by DV 2.0 design.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
import uuid as _uuid

dbutils.widgets.text("CATALOG",       "workspace",  "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA",  "vault",      "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver",     "Silver schema")
dbutils.widgets.text("MODEL_PATH",    "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS",  "",           "Optional: only process records >= this timestamp")
dbutils.widgets.text("ALERT_CHANNEL", "log",        "log | webhook | all")
dbutils.widgets.text("WEBHOOK_URL",   "",           "Optional Slack/Teams webhook")

CATALOG        = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA   = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA  = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH     = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS   = dbutils.widgets.get("WATERMARK_TS")
ALERT_CHANNEL  = dbutils.widgets.get("ALERT_CHANNEL")
WEBHOOK_URL    = dbutils.widgets.get("WEBHOOK_URL") or None
VAULT_RUN_ID   = str(_uuid.uuid4())

In [ ]:
import uuid as _uuid

dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Silver schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS", "", "Optional: only process records >= this timestamp")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS  = dbutils.widgets.get("WATERMARK_TS")
VAULT_RUN_ID  = str(_uuid.uuid4())

In [ ]:
model = load_model(MODEL_PATH)
print(f"Loaded {len(model['hubs'])} hubs from model")

In [ ]:
# ---------------------------------------------------------------------------
# Task 3.1 — Hub DQ: HK uniqueness per hub
# Task 4.3 — Alert on FAIL
# ---------------------------------------------------------------------------
print('\nHub ingestion complete.')
dq_failures = []

for hub_cfg in model['hubs']:
    if not hub_cfg.get('enabled', True):
        continue

    hub_name   = hub_cfg['name']
    hk_alias   = f"HK_{hub_name[4:]}"
    target_tbl = f"{CATALOG}.{VAULT_SCHEMA}.{hub_name.lower()}"
    cnt        = hub_counts.get(hub_name, 0)
    print(f'  {hub_name}: {cnt:,} rows')

    row = spark.sql(f"""
        SELECT COUNT(*) AS total, COUNT(DISTINCT {hk_alias}) AS uniq FROM {target_tbl}
    """).collect()[0]
    dups   = row["total"] - row["uniq"]
    status = "PASS" if dups == 0 else "FAIL"

    write_dq_result(
        spark, run_id=VAULT_RUN_ID, layer="vault", table_name=hub_name.lower(),
        check_name="hk_uniqueness", status=status,
        observed_value=float(dups), threshold=0.0,
        message=f"{dups} duplicate HK(s)" if dups else None,
    )
    if status == "FAIL":
        alert_dq_failure(
            table_name=hub_name.lower(), check_name="hk_uniqueness",
            observed_value=dups, layer="vault",
            alert_channel=ALERT_CHANNEL, webhook_url=WEBHOOK_URL,
        )
        dq_failures.append(f"{hub_name}: {dups} duplicate HK(s)")

if dq_failures:
    raise RuntimeError("Hub DQ FAIL:\n" + "\n".join(dq_failures))

print("Hub DQ checks PASSED.")

In [ ]:
# ---------------------------------------------------------------------------
# Task 3.1 — Hub DQ: HK uniqueness per hub
# ---------------------------------------------------------------------------
print('\nHub ingestion complete.')
dq_failures = []

for hub_cfg in model['hubs']:
    if not hub_cfg.get('enabled', True):
        continue

    hub_name  = hub_cfg['name']
    hk_alias  = f"HK_{hub_name[4:]}"
    target_tbl = f"{CATALOG}.{VAULT_SCHEMA}.{hub_name.lower()}"

    cnt = hub_counts.get(hub_name, 0)
    print(f'  {hub_name}: {cnt:,} rows')

    # HK uniqueness — hubs must never have duplicate hash keys
    row = spark.sql(f"""
        SELECT COUNT(*) AS total, COUNT(DISTINCT {hk_alias}) AS uniq
        FROM {target_tbl}
    """).collect()[0]
    dups = row["total"] - row["uniq"]
    status = "PASS" if dups == 0 else "FAIL"
    write_dq_result(
        spark, run_id=VAULT_RUN_ID, layer="vault", table_name=hub_name.lower(),
        check_name="hk_uniqueness", status=status,
        observed_value=float(dups), threshold=0.0,
        message=f"{dups} duplicate HK(s)" if dups else None,
    )
    if status == "FAIL":
        dq_failures.append(f"{hub_name}: {dups} duplicate HK(s)")

if dq_failures:
    raise RuntimeError(f"Hub DQ FAIL:\n" + "\n".join(dq_failures))

print("Hub DQ checks PASSED.")